# E39 unscored T4 deployment probe

This private Notebook checks whether the fixed E24 + 1×E39 pipeline can load and process 4,000 **unlabelled training-side** texts under Internet off and Kaggle T4×2. It uses an existing fold-1 E39 adapter, not a full submission adapter. It writes synthetic IDs to a nested Parquet file for schema testing. Do not click **Submit** on this version. The original E24 Notebook remains the submitted baseline.


In [ ]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
COMPETITION_DIR = INPUT_ROOT / "competitions" / "name-that-llm"
ARTIFACT_ROOT = None  # Set to the mounted Kaggle Dataset directory if autodetection is ambiguous.
ARTIFACT_ZIP = None  # Optional path to an attached ZIP if multiple model ZIPs are mounted.
WHEELHOUSE_DIR = None  # Auto-filled from an attached extracted wheelhouse directory or ZIP.
WHEELHOUSE_ZIP = None  # Optional explicit ZIP path if autodetection is ambiguous.
OUTPUT_PATH = Path("/kaggle/working/submission.parquet")
MODERN_ARTIFACT_ROOT = None  # Optional mounted E16 Dataset directory override.
MODERN_ARTIFACT_ZIP = None  # Optional E16 ZIP override if autodetection is ambiguous.

E24_ARTIFACT_ROOT = None  # Optional mounted E24 Dataset directory override.
E24_ARTIFACT_ZIP = None  # Optional E24 ZIP override if autodetection is ambiguous.

import sys
import time
PROBE_STARTED = time.monotonic()
PROBE_ROOT = Path('/kaggle/working/e39_unscored_probe')
OUTPUT_PATH = PROBE_ROOT / 'submission.parquet'  # Never the competition output path.
BASE_DIR = None  # Pinned Qwen3-8B-Base Dataset directory; auto-detect if unique.
ADAPTER_DIR = None  # Private fold-1 E39 adapter directory; auto-detect if unique.
SMOKE_REFERENCE_PATH = None  # Private, train-side 3090 reference JSON.
GPU_PYTHON = Path(sys.executable)  # Set to verified offline GPU venv if needed.
DEVICE_MODE = 'single'  # Fixed first test; use 'dual' only after single-T4 OOM.
TIME_LIMIT_SECONDS = 8 * 3600  # One hour below the competition's nine-hour cap.


In [ ]:
import hashlib
import json
import zipfile

RECIPE = "e0c_plus_roberta_12ep_scalerfix_equal_rank"
SOURCE_FILES = ("final_pipeline.py", "run_e0.py", "run_style.py",
                "run_combined.py", "run_fixed_blend.py")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

ARTIFACT_ARCHIVE = None
if ARTIFACT_ROOT is None:
    candidates = []
    for item in INPUT_ROOT.rglob("artifact_manifest.json"):
        try:
            candidate = json.loads(item.read_text(encoding="utf-8"))
        except (OSError, ValueError):
            continue
        if candidate.get("recipe") == RECIPE:
            candidates.append(item.parent)
    if not candidates:
        zip_candidates = []
        for item in INPUT_ROOT.rglob("*.zip"):
            try:
                with zipfile.ZipFile(item) as bundle:
                    if "artifact_manifest.json" in bundle.namelist():
                        zip_candidates.append(item)
            except (OSError, zipfile.BadZipFile):
                continue
        if ARTIFACT_ZIP is not None:
            zip_candidates = [Path(ARTIFACT_ZIP)]
        if len(zip_candidates) != 1:
            raise RuntimeError(
                f"Expected one model artifact ZIP, found {len(zip_candidates)}: {zip_candidates}. "
                "Set ARTIFACT_ZIP in the config cell."
            )
        ARTIFACT_ARCHIVE = Path(zip_candidates[0]).resolve(strict=True)
        extracted_root = Path("/kaggle/working/name_that_llm_artifact")
        with zipfile.ZipFile(zip_candidates[0]) as bundle:
            names = bundle.namelist()
            if len(names) != len(set(names)) or "artifact_manifest.json" not in names:
                raise RuntimeError("Invalid model artifact ZIP")
            if any(name.startswith("/") or ".." in name.split("/") or "\\" in name for name in names):
                raise RuntimeError("Unsafe path in model artifact ZIP")
            bundle.extractall(extracted_root)
        candidates = [extracted_root]
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one model artifact Dataset, found {len(candidates)}: {candidates}. "
            "Set ARTIFACT_ROOT in the config cell."
        )
    ARTIFACT_ROOT = candidates[0]
ARTIFACT_ROOT = Path(ARTIFACT_ROOT).resolve(strict=True)
manifest = json.loads((ARTIFACT_ROOT / "artifact_manifest.json").read_text(encoding="utf-8"))
if manifest.get("format_version") != 1 or manifest.get("recipe") != RECIPE:
    raise RuntimeError("Unexpected artifact manifest")
for relative_name, expected in manifest["sha256"].items():
    path = (ARTIFACT_ROOT / relative_name).resolve(strict=True)
    if not path.is_relative_to(ARTIFACT_ROOT) or sha256_file(path) != expected:
        raise RuntimeError(f"Artifact hash mismatch: {relative_name}")

SOURCE_DIR = (ARTIFACT_ROOT / manifest["source_dir"]).resolve(strict=True)
PACKAGE_DIR = (ARTIFACT_ROOT / manifest["package_dir"]).resolve(strict=True)
if not SOURCE_DIR.is_relative_to(ARTIFACT_ROOT) or not PACKAGE_DIR.is_relative_to(ARTIFACT_ROOT):
    raise RuntimeError("Artifact source/package directory escapes the mounted Dataset")
metadata = json.loads((PACKAGE_DIR / "metadata.json").read_text(encoding="utf-8"))
if metadata.get("recipe") != RECIPE or metadata.get("mode") != "full":
    raise RuntimeError("Expected the trained full package")
if set(metadata.get("source_sha256", {})) != set(SOURCE_FILES):
    raise RuntimeError("Unexpected source manifest in model package")
for name in SOURCE_FILES:
    if sha256_file(SOURCE_DIR / name) != metadata["source_sha256"][name]:
        raise RuntimeError(f"Source differs from model training: {name}")
TEST_PATH = COMPETITION_DIR / "test.csv"
if not TEST_PATH.is_file():
    raise FileNotFoundError(f"Official competition test file missing: {TEST_PATH}")
print("Verified full model package and source hashes:", ARTIFACT_ROOT)

In [ ]:
import json
import zipfile

MODERN_RECIPE = 'e16_three_seed_modernbert_plus_e0c_roberta_row_z'
MODERN_MANIFEST_NAME = 'e16_artifact_manifest.json'
MODERN_ARTIFACT_ARCHIVE = None

if MODERN_ARTIFACT_ROOT is None:
    candidates = []
    for item in INPUT_ROOT.rglob(MODERN_MANIFEST_NAME):
        try:
            candidate = json.loads(item.read_text(encoding="utf-8"))
        except (OSError, ValueError):
            continue
        if candidate.get("recipe") == MODERN_RECIPE:
            candidates.append(item.parent)
    if not candidates:
        zip_candidates = []
        for item in INPUT_ROOT.rglob("*.zip"):
            try:
                with zipfile.ZipFile(item) as bundle:
                    if MODERN_MANIFEST_NAME in bundle.namelist():
                        zip_candidates.append(item)
            except (OSError, zipfile.BadZipFile):
                continue
        if MODERN_ARTIFACT_ZIP is not None:
            zip_candidates = [Path(MODERN_ARTIFACT_ZIP)]
        if len(zip_candidates) != 1:
            raise RuntimeError(
                f"Expected one E16 artifact ZIP, found {len(zip_candidates)}: {zip_candidates}. "
                "Set MODERN_ARTIFACT_ZIP in the config cell."
            )
        MODERN_ARTIFACT_ARCHIVE = Path(zip_candidates[0]).resolve(strict=True)
        extracted_root = Path("/kaggle/working/name_that_llm_e16_artifact")
        with zipfile.ZipFile(MODERN_ARTIFACT_ARCHIVE) as bundle:
            names = bundle.namelist()
            if len(names) != len(set(names)) or MODERN_MANIFEST_NAME not in names:
                raise RuntimeError("Invalid E16 artifact ZIP")
            if any(name.startswith("/") or ".." in name.split("/") or "\\" in name for name in names):
                raise RuntimeError("Unsafe path in E16 artifact ZIP")
            bundle.extractall(extracted_root)
        candidates = [extracted_root]
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one E16 Dataset, found {len(candidates)}: {candidates}. "
            "Set MODERN_ARTIFACT_ROOT in the config cell."
        )
    MODERN_ARTIFACT_ROOT = candidates[0]

MODERN_ARTIFACT_ROOT = Path(MODERN_ARTIFACT_ROOT).resolve(strict=True)
modern_manifest = json.loads(
    (MODERN_ARTIFACT_ROOT / MODERN_MANIFEST_NAME).read_text(encoding="utf-8")
)
if modern_manifest.get("format_version") != 1 or modern_manifest.get("recipe") != MODERN_RECIPE:
    raise RuntimeError("Unexpected E16 artifact manifest")
if modern_manifest.get("source_file") != "source/fit_modernbert_full.py":
    raise RuntimeError("Unexpected E16 source path")
expected_package_dirs = {"42": "modern_seed42", "17": "modern_seed17", "73": "modern_seed73"}
if modern_manifest.get("package_dirs") != expected_package_dirs:
    raise RuntimeError("Unexpected E16 package directory mapping")
modern_hashes = modern_manifest.get("sha256")
if not isinstance(modern_hashes, dict) or not modern_hashes:
    raise RuntimeError("E16 artifact file hash manifest is missing")
actual_files = {item.relative_to(MODERN_ARTIFACT_ROOT).as_posix()
                for item in MODERN_ARTIFACT_ROOT.rglob("*") if item.is_file()}
if actual_files != set(modern_hashes) | {MODERN_MANIFEST_NAME}:
    raise RuntimeError("E16 artifact files differ from manifest")
for relative_name, expected_hash in modern_hashes.items():
    path = (MODERN_ARTIFACT_ROOT / relative_name).resolve(strict=True)
    if not path.is_relative_to(MODERN_ARTIFACT_ROOT) or sha256_file(path) != expected_hash:
        raise RuntimeError(f"E16 artifact hash mismatch: {relative_name}")

MODERN_SOURCE_PATH = (MODERN_ARTIFACT_ROOT / modern_manifest["source_file"]).resolve(strict=True)
MODERN_PACKAGE_DIRS = {
    int(seed): (MODERN_ARTIFACT_ROOT / directory).resolve(strict=True)
    for seed, directory in expected_package_dirs.items()
}
for seed, package in MODERN_PACKAGE_DIRS.items():
    if not package.is_relative_to(MODERN_ARTIFACT_ROOT):
        raise RuntimeError(f"E16 seed {seed} package escapes Dataset root")
    trained = json.loads((package / "metadata.json").read_text(encoding="utf-8"))
    if trained.get("seed") != seed:
        raise RuntimeError(f"E16 seed directory/metadata mismatch: {seed}")
    if trained.get("source_sha256", {}).get("fit_modernbert_full.py") != sha256_file(MODERN_SOURCE_PATH):
        raise RuntimeError(f"E16 seed {seed} source differs from fitted source")
    smoke = json.loads((package / "smoke_verification.json").read_text(encoding="utf-8"))
    if (smoke.get("seed") != seed or smoke.get("all_predictions_equal") is not True
            or smoke.get("package_metadata_sha256") != sha256_file(package / "metadata.json")
            or not isinstance(smoke.get("max_abs_logit_difference"), (int, float))
            or not 0 <= smoke["max_abs_logit_difference"] <= 2e-5):
        raise RuntimeError(f"E16 seed {seed} has no valid reload smoke verification")
print("Verified E16 ModernBERT package/source hashes:", MODERN_ARTIFACT_ROOT)


In [ ]:
import json
import zipfile

E24_RECIPE = 'e24_audited_prvsiyan_v9_full_e0c_replacement'
E24_MANIFEST_NAME = 'e24_artifact_manifest.json'
if E24_ARTIFACT_ROOT is None:
    candidates = []
    for item in INPUT_ROOT.rglob(E24_MANIFEST_NAME):
        try:
            candidate = json.loads(item.read_text(encoding="utf-8"))
        except (OSError, ValueError):
            continue
        if candidate.get("recipe") == E24_RECIPE:
            candidates.append(item.parent)
    if not candidates:
        zip_candidates = []
        for item in INPUT_ROOT.rglob("*.zip"):
            try:
                with zipfile.ZipFile(item) as bundle:
                    if E24_MANIFEST_NAME in bundle.namelist():
                        zip_candidates.append(item)
            except (OSError, zipfile.BadZipFile):
                continue
        if E24_ARTIFACT_ZIP is not None:
            zip_candidates = [Path(E24_ARTIFACT_ZIP)]
        if len(zip_candidates) != 1:
            raise RuntimeError(f"Expected one E24 artifact ZIP, found {len(zip_candidates)}: {zip_candidates}")
        extracted = Path("/kaggle/working/name_that_llm_e24_artifact")
        with zipfile.ZipFile(zip_candidates[0]) as bundle:
            names = bundle.namelist()
            if (len(names) != len(set(names)) or E24_MANIFEST_NAME not in names
                    or any(name.startswith("/") or ".." in name.split("/") or "\\" in name
                           for name in names)):
                raise RuntimeError("Invalid E24 artifact ZIP")
            bundle.extractall(extracted)
        candidates = [extracted]
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one E24 Dataset, found {len(candidates)}: {candidates}")
    E24_ARTIFACT_ROOT = candidates[0]

E24_ARTIFACT_ROOT = Path(E24_ARTIFACT_ROOT).resolve(strict=True)
e24_manifest = json.loads((E24_ARTIFACT_ROOT / E24_MANIFEST_NAME).read_text(encoding="utf-8"))
if (e24_manifest.get("format_version") != 1 or e24_manifest.get("recipe") != E24_RECIPE
        or e24_manifest.get("package_dir") != "e24_e0c"
        or e24_manifest.get("source_dir") != "source"):
    raise RuntimeError("Unexpected E24 artifact manifest")
e24_hashes = e24_manifest.get("sha256")
if not isinstance(e24_hashes, dict) or set(e24_hashes) != {
    "e24_e0c/metadata.json", "e24_e0c/e0c.joblib", "e24_e0c/smoke_reference.json",
    "source/fit_e24_full_e0c.py", "source/run_e24_prvsiyan_external_ablation.py",
}:
    raise RuntimeError("Unexpected E24 artifact file list")
actual_files = {p.relative_to(E24_ARTIFACT_ROOT).as_posix()
                for p in E24_ARTIFACT_ROOT.rglob("*") if p.is_file()}
if actual_files != set(e24_hashes) | {E24_MANIFEST_NAME}:
    raise RuntimeError("E24 artifact contents differ from manifest")
for name, digest in e24_hashes.items():
    path = (E24_ARTIFACT_ROOT / name).resolve(strict=True)
    if not path.is_relative_to(E24_ARTIFACT_ROOT) or sha256_file(path) != digest:
        raise RuntimeError(f"E24 artifact hash mismatch: {name}")
E24_PACKAGE_DIR = (E24_ARTIFACT_ROOT / "e24_e0c").resolve(strict=True)
E24_SOURCE_DIR = (E24_ARTIFACT_ROOT / "source").resolve(strict=True)
trained = json.loads((E24_PACKAGE_DIR / "metadata.json").read_text(encoding="utf-8"))
for name in ("fit_e24_full_e0c.py", "run_e24_prvsiyan_external_ablation.py"):
    if sha256_file(E24_SOURCE_DIR / name) != trained.get("source_sha256", {}).get(name):
        raise RuntimeError(f"E24 fitted source mismatch: {name}")
for name in ("run_e0.py", "run_style.py", "run_combined.py"):
    if sha256_file(SOURCE_DIR / name) != trained.get("source_sha256", {}).get(name):
        raise RuntimeError(f"E24 shared E0c source mismatch: {name}")
print("Verified small E24 E0c package/source hashes:", E24_ARTIFACT_ROOT)


In [ ]:
import importlib.metadata as importlib_metadata
import os
import platform
import shutil
import subprocess
import sys

WHEELHOUSE_ARCHIVE_SHA256 = "2c4dc134de5717ed8d974c4fb8405f44d848045c08b08706826436ec66480a84"

def installed_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None

def runtime_issues():
    issues = []
    exact = {
        "scikit-learn": metadata["sklearn_version"],
        "torch": metadata["torch_version"].split("+")[0],
        "transformers": metadata["transformers_version"],
    }
    for name, expected in exact.items():
        actual = installed_version(name)
        comparable = actual.split("+")[0] if actual and name == "torch" else actual
        if comparable != expected:
            issues.append(f"{name}: installed={actual}, required={expected}")
    for name in ("numpy", "pandas", "joblib", "safetensors", "pyarrow"):
        if installed_version(name) is None:
            issues.append(f"{name}: missing")
    return issues

PYTHON_EXECUTABLE = Path(sys.executable)
issues = runtime_issues()
if issues:
    if WHEELHOUSE_DIR is None:
        directory_candidates = [
            item.parent for item in INPUT_ROOT.rglob("requirements-kaggle-offline.txt")
            if (item.parent / "manifest.json").is_file()
        ]
        if len(directory_candidates) > 1:
            siblings = [item for item in directory_candidates if item.parent == ARTIFACT_ROOT.parent]
            if siblings:
                directory_candidates = siblings
        if len(directory_candidates) == 1:
            WHEELHOUSE_DIR = directory_candidates[0]
            print("Found extracted wheelhouse Dataset:", WHEELHOUSE_DIR)
        elif directory_candidates:
            raise RuntimeError(
                f"Found multiple wheelhouse directories: {directory_candidates}. "
                "Set WHEELHOUSE_DIR in the config cell."
            )
        else:
            if WHEELHOUSE_ZIP is None:
                wheelhouse_candidates = list(INPUT_ROOT.rglob("kaggle_wheelhouse_cp313_linux_x86_64.zip"))
                if ARTIFACT_ARCHIVE is not None:
                    siblings = [item for item in wheelhouse_candidates if item.parent == ARTIFACT_ARCHIVE.parent]
                    if siblings:
                        wheelhouse_candidates = siblings
                if len(wheelhouse_candidates) != 1:
                    raise RuntimeError(
                        f"Dependency mismatch: {issues}. Found {len(wheelhouse_candidates)} wheelhouse ZIPs: "
                        f"{wheelhouse_candidates}. Set WHEELHOUSE_DIR or WHEELHOUSE_ZIP."
                    )
                wheelhouse_zip = wheelhouse_candidates[0]
            else:
                wheelhouse_zip = Path(WHEELHOUSE_ZIP)
            wheelhouse_zip = wheelhouse_zip.resolve(strict=True)
            if sha256_file(wheelhouse_zip) != WHEELHOUSE_ARCHIVE_SHA256:
                raise RuntimeError("Wheelhouse ZIP SHA-256 differs from the validated upload")
            extracted_wheels = Path("/kaggle/working/name_that_llm_wheelhouse")
            with zipfile.ZipFile(wheelhouse_zip) as bundle:
                names = bundle.namelist()
                if len(names) != len(set(names)) or any(
                    "/" in name or "\\" in name or name in (".", "..") for name in names
                ):
                    raise RuntimeError("Unsafe or duplicate path in wheelhouse ZIP")
                bundle.extractall(extracted_wheels)
            WHEELHOUSE_DIR = extracted_wheels
            print("Extracted offline wheelhouse:", WHEELHOUSE_DIR)
    WHEELHOUSE_DIR = Path(WHEELHOUSE_DIR).resolve(strict=True)
    if not WHEELHOUSE_DIR.is_dir():
        raise NotADirectoryError(WHEELHOUSE_DIR)
    if sys.version_info[:2] != (3, 13) or sys.platform != "linux" or platform.machine() != "x86_64":
        raise RuntimeError(
            "Attached wheelhouse requires CPython 3.13 on Linux x86_64; "
            f"found Python {sys.version.split()[0]} on {sys.platform}/{platform.machine()}"
        )
    manifest_rows = json.loads((WHEELHOUSE_DIR / "manifest.json").read_text(encoding="utf-8"))
    expected_wheels = {row["file"]: row for row in manifest_rows}
    expected_names = set(expected_wheels) | {
        "manifest.json", "README.txt", "SHA256SUMS.txt", "requirements-kaggle-offline.txt"
    }
    if (
        len(manifest_rows) != 35
        or len(expected_wheels) != 35
        or any(not name.endswith(".whl") or "/" in name or "\\" in name for name in expected_wheels)
        or {item.name for item in WHEELHOUSE_DIR.iterdir()} != expected_names
    ):
        raise RuntimeError("Wheelhouse directory layout differs from its 35-wheel manifest")
    for name, item in expected_wheels.items():
        wheel = WHEELHOUSE_DIR / name
        if wheel.stat().st_size != item["bytes"] or sha256_file(wheel) != item["sha256"]:
            raise RuntimeError(f"Wheelhouse integrity check failed: {name}")
    print("Verified all 35 offline wheel hashes")
    requirement_file = WHEELHOUSE_DIR / "requirements-kaggle-offline.txt"
    if not requirement_file.is_file() or "torch==2.6.0+cpu" not in requirement_file.read_text(encoding="utf-8"):
        raise RuntimeError("Expected the validated CPU-only Kaggle wheelhouse requirements")
    # Kaggle's system Python may fail ensurepip. Build a pip-free venv, then
    # let a host pip >=22.3 manage that interpreter (pip's documented --python option).
    venv_dir = Path("/kaggle/working/name_that_llm_venv_nodeps")
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(venv_dir)], check=True)
    venv_python = venv_dir / "bin" / "python"
    if not venv_python.is_file():
        raise RuntimeError(f"Virtual environment interpreter missing: {venv_python}")
    pip_frontend = [sys.executable, "-m", "pip"]
    pip_probe = subprocess.run(pip_frontend + ["--version"], capture_output=True, text=True)
    if pip_probe.returncode:
        pip_command = shutil.which("pip")
        if pip_command is None:
            raise RuntimeError("No system pip is available to bootstrap the pip-free venv offline")
        pip_frontend = [pip_command]
    # Install every verified wheel explicitly. The hub package declares hf-xet
    # for remote transfers, but that wheel is absent and inference is local-only.
    wheel_paths = [str(WHEELHOUSE_DIR / name) for name in sorted(expected_wheels)]
    subprocess.run(
        pip_frontend + ["--python", str(venv_dir), "install", "--no-index", "--no-deps"]
        + wheel_paths,
        check=True,
    )
    probe = (
        "import sys, numpy, pandas, sklearn, joblib, torch, transformers, pyarrow, safetensors; "
        "from transformers import AutoConfig, AutoTokenizer; "
        f"assert sklearn.__version__ == {metadata['sklearn_version']!r}; "
        f"assert torch.__version__.split('+')[0] == {metadata['torch_version'].split('+')[0]!r}; "
        f"assert transformers.__version__ == {metadata['transformers_version']!r}; "
        "AutoConfig.from_pretrained(sys.argv[1], local_files_only=True); "
        "AutoTokenizer.from_pretrained(sys.argv[2], local_files_only=True, use_fast=True); "
        "print('Offline venv and local model loading ready:', torch.__version__)"
    )
    probe_environment = os.environ.copy()
    probe_environment["HF_HUB_OFFLINE"] = "1"
    probe_environment["TRANSFORMERS_OFFLINE"] = "1"
    probe_environment["HF_HUB_DISABLE_XET"] = "1"
    subprocess.run(
        [str(venv_python), "-c", probe, str(PACKAGE_DIR / "encoder_config"),
         str(PACKAGE_DIR / "tokenizer")],
        env=probe_environment, check=True,
    )
    PYTHON_EXECUTABLE = venv_python
else:
    print("Notebook runtime already matches the fitted package; using", PYTHON_EXECUTABLE)


In [ ]:
# Probe the ModernBERT config and tokenizer with the interpreter selected above.
probe_environment = os.environ.copy()
probe_environment["HF_HUB_OFFLINE"] = "1"
probe_environment["TRANSFORMERS_OFFLINE"] = "1"
probe_environment["HF_HUB_DISABLE_XET"] = "1"
probe_environment["TORCHDYNAMO_DISABLE"] = "1"
probe = (
    "import sys, transformers; from transformers import AutoConfig, AutoTokenizer; "
    "c=AutoConfig.from_pretrained(sys.argv[1], local_files_only=True); "
    "assert c.model_type == 'modernbert' and c.hidden_size == 768; "
    "AutoTokenizer.from_pretrained(sys.argv[2], local_files_only=True, use_fast=True); "
    "print('Local ModernBERT config and tokenizer ready:', transformers.__version__)"
)
subprocess.run(
    [str(PYTHON_EXECUTABLE), "-c", probe,
     str(MODERN_PACKAGE_DIRS[42] / "encoder_config"),
     str(MODERN_PACKAGE_DIRS[42] / "tokenizer")],
    env=probe_environment, check=True,
)


In [ ]:
import hashlib
import os
import shutil
import subprocess

WORK_SOURCE = Path('/kaggle/working/name_that_llm_source_e39_probe')
shutil.copytree(SOURCE_DIR, WORK_SOURCE, dirs_exist_ok=True)
shutil.copy2(MODERN_SOURCE_PATH, WORK_SOURCE / 'fit_modernbert_full.py')
for name in ('fit_e24_full_e0c.py', 'run_e24_prvsiyan_external_ablation.py'):
    shutil.copy2(E24_SOURCE_DIR / name, WORK_SOURCE / name)
WORK_KAGGLE = WORK_SOURCE / 'kaggle'
WORK_KAGGLE.mkdir(parents=True, exist_ok=True)
SOURCE_TEXT = {'submit_e24_external_e0c.py': '#!/usr/bin/env python3\n"""Offline E24 submission: audited external E0c plus unchanged E16 neural streams.\n\nEvery component is standardized across its 18 class scores within each row.\nThe ModernBERT stream is the mean of three separately standardized seeds;\nthe final score is the equal-weight sum of the three model-family streams.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport hashlib\nimport json\nimport sys\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport sklearn\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom final_pipeline import load_test, predict_package  # noqa: E402\nfrom fit_modernbert_full import (  # noqa: E402\n    LABELS as MODERN_LABELS,\n    SMOKE_TEXTS,\n    load_package as load_modern_package,\n    score_texts as score_modern_texts,\n)\nfrom run_e0 import LABELS  # noqa: E402\n\n\nEXPECTED_SEEDS = (42, 17, 73)\nEXPECTED_TRAIN_SHA256 = "40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707"\nEXPECTED_MODEL_ID = "answerdotai/ModernBERT-base"\nEXPECTED_REVISION = "8949b909ec900327062f0ebf497f51aef5e6f0c8"\nE24_RECIPE = "e24_audited_prvsiyan_v9_full_e0c_replacement"\nE24_SOURCE_FILES = (\n    "fit_e24_full_e0c.py", "run_e24_prvsiyan_external_ablation.py",\n    "run_e0.py", "run_style.py", "run_combined.py",\n)\n# Package creation already requires <=2e-5 on the training machine. The\n# submission may run on a different CPU build; keep a separate cross-runtime\n# smoke tolerance while still requiring identical argmax and all file hashes.\nCROSS_RUNTIME_SMOKE_ATOL = 1e-3\n\n\ndef sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef load_e24_surface(path: Path, source_dir: Path):\n    """Check exact fitted source/data provenance and three-text reload smoke."""\n    path = path.resolve(strict=True)\n    metadata = json.loads((path / "metadata.json").read_text(encoding="utf-8"))\n    required = {\n        "package_schema_version": 1,\n        "recipe": E24_RECIPE,\n        "train_sha256": EXPECTED_TRAIN_SHA256,\n        "n_official_rows": 2699,\n        "n_official_prompts": 150,\n        "n_external_rows": 3620,\n        "n_external_prompts": 250,\n        "labels": list(LABELS),\n        "external_to_official_weight_ratio": 0.25,\n        "selected_manifest_sha256": "d7f8b7c9b0480ec896777e667589bf7d2fe12e503aa8f867c0708d0656ad6191",\n    }\n    for key, value in required.items():\n        if metadata.get(key) != value:\n            raise ValueError(f"E24 E0c has unexpected {key}")\n    if metadata.get("sklearn_version") != sklearn.__version__:\n        raise ValueError("E24 E0c requires its fitted scikit-learn version")\n    counts = metadata.get("official_rows_by_class", {})\n    external_counts = metadata.get("external_rows_by_class", {})\n    totals = metadata.get("external_total_weight_by_class", {})\n    if (sum(counts.values()) != 2699 or counts.get("claude-fable-5") != 149\n            or any(counts.get(label) != 150 for label in LABELS if label != "claude-fable-5")\n            or sum(external_counts.values()) != 3620\n            or external_counts.get("qwen3.8-max") != 0):\n        raise ValueError("E24 E0c class counts differ from locked full fit")\n    for label in LABELS:\n        expected = 0.0 if label == "qwen3.8-max" else 0.25 * counts[label]\n        if not np.isclose(totals.get(label, np.nan), expected, atol=1e-10):\n            raise ValueError(f"E24 E0c external weight ratio failed for {label}")\n    sources = metadata.get("source_sha256", {})\n    if set(sources) != set(E24_SOURCE_FILES):\n        raise ValueError("E24 fitted source list changed")\n    for name in E24_SOURCE_FILES:\n        if sha256(source_dir / name) != sources[name]:\n            raise ValueError(f"E24 fitted source hash changed: {name}")\n    hashes = metadata.get("artifact_sha256", {})\n    if set(hashes) != {"e0c.joblib", "smoke_reference.json"}:\n        raise ValueError("E24 E0c artifact hash list changed")\n    for name, digest in hashes.items():\n        if sha256(path / name) != digest:\n            raise ValueError(f"E24 E0c file hash changed: {name}")\n    model = joblib.load(path / "e0c.joblib")\n    classes = model.named_steps["classifier"].classes_\n    if set(classes) != set(LABELS):\n        raise ValueError("E24 E0c lost an official class")\n    positions = [int(np.flatnonzero(classes == label)[0]) for label in LABELS]\n    reference = json.loads((path / "smoke_reference.json").read_text(encoding="utf-8"))\n    if reference.get("labels") != list(LABELS) or len(reference.get("texts", [])) != 3:\n        raise ValueError("E24 E0c smoke reference changed")\n    expected = np.asarray(reference.get("logits"), dtype=np.float64)\n    actual = np.asarray(model.decision_function(reference["texts"])[:, positions], dtype=np.float64)\n    if expected.shape != (3, len(LABELS)) or not np.isfinite(actual).all():\n        raise ValueError("E24 E0c smoke logits invalid")\n    if not np.allclose(actual, expected, rtol=0, atol=1e-5):\n        raise ValueError("E24 E0c smoke logits changed after reload")\n    if not np.array_equal(actual.argmax(axis=1), expected.argmax(axis=1)):\n        raise ValueError("E24 E0c smoke predictions changed after reload")\n    return model, positions\n\n\ndef row_standardize(scores: np.ndarray, n_rows: int) -> np.ndarray:\n    array = np.asarray(scores, dtype=np.float64)\n    if array.shape != (n_rows, len(LABELS)) or not np.isfinite(array).all():\n        raise ValueError(f"Invalid or non-finite 18-class scores: {array.shape}")\n    centered = array - array.mean(axis=1, keepdims=True)\n    scale = centered.std(axis=1, keepdims=True)\n    if (scale <= 0).any() or not np.isfinite(scale).all():\n        raise ValueError("Constant or non-finite 18-class score row")\n    return centered / scale\n\n\ndef validate_modern_metadata(path: Path, metadata: dict, expected_seed: int) -> None:\n    if not isinstance(metadata, dict):\n        raise ValueError(f"ModernBERT metadata is not an object: {path}")\n    required = {\n        "package_schema_version": 1,\n        "mode": "full",\n        "seed": expected_seed,\n        "model_id": EXPECTED_MODEL_ID,\n        "requested_revision": EXPECTED_REVISION,\n        "pretrained_revision": EXPECTED_REVISION,\n        "train_sha256": EXPECTED_TRAIN_SHA256,\n        "n_training_rows": 2699,\n        "n_training_prompts": 150,\n        "labels": list(LABELS),\n    }\n    for key, expected in required.items():\n        if metadata.get(key) != expected:\n            raise ValueError(f"ModernBERT seed {expected_seed} has unexpected {key}: {path}")\n    if not metadata.get("artifact_sha256") or not metadata.get("source_sha256"):\n        raise ValueError(f"ModernBERT package hashes missing: {path}")\n    config = metadata.get("training_config", {})\n    for key, expected in (\n        ("epochs", 12), ("max_length", 256), ("batch_size", 16),\n        ("dropout", 0.1), ("pooling", "mean"),\n    ):\n        if config.get(key) != expected:\n            raise ValueError(f"ModernBERT seed {expected_seed} has unexpected {key}: {path}")\n    runtime = metadata.get("runtime_versions", {})\n    if runtime.get("torch", "").split("+")[0] != "2.6.0" or runtime.get("transformers") != "4.51.3":\n        raise ValueError(f"ModernBERT seed {expected_seed} runtime differs from fitted runtime")\n\n\ndef submit(\n    full_package: Path,\n    modern_packages: tuple[Path, Path, Path],\n    e24_package: Path,\n    test_path: Path,\n    output_path: Path,\n    batch_size: int = 16,\n) -> None:\n    if output_path.exists():\n        raise ValueError(f"Submission already exists: {output_path}")\n    if batch_size < 1:\n        raise ValueError("Batch size must be positive")\n    if len(MODERN_LABELS) != len(LABELS) or list(MODERN_LABELS) != list(LABELS):\n        raise ValueError("ModernBERT label order differs from the official fixed label order")\n    paths = tuple(path.resolve(strict=True) for path in modern_packages)\n    if len(set(paths)) != len(paths):\n        raise ValueError("Each ModernBERT seed must use a different package directory")\n    for expected_seed, path in zip(EXPECTED_SEEDS, paths, strict=True):\n        with (path / "metadata.json").open("r", encoding="utf-8") as handle:\n            declared_metadata = json.load(handle)\n        validate_modern_metadata(path, declared_metadata, expected_seed)\n    old_metadata = json.loads((full_package / "metadata.json").read_text(encoding="utf-8"))\n    for key, expected in (\n        ("train_sha256", EXPECTED_TRAIN_SHA256),\n        ("n_training_rows", 2699),\n        ("n_training_prompts", 150),\n    ):\n        if old_metadata.get(key) != expected:\n            raise ValueError(f"E0c/RoBERTa package has unexpected {key}")\n\n    test = load_test(test_path)\n    texts = test["text"].tolist()\n    n_rows = len(test)\n    print(f"Scoring {n_rows} official test rows", flush=True)\n\n    e24_surface, e24_positions = load_e24_surface(e24_package, Path(__file__).resolve().parents[1])\n    e24_logits = np.asarray(e24_surface.decision_function(texts)[:, e24_positions], dtype=np.float64)\n    _, old_surface_logits, roberta_logits = predict_package(full_package, texts, "full")\n    if old_surface_logits.shape != e24_logits.shape:\n        raise ValueError("E24 E0c and original E0c class score shapes differ")\n    family_score = row_standardize(e24_logits, n_rows)\n    family_score += row_standardize(roberta_logits, n_rows)\n    del e24_surface, e24_logits, old_surface_logits, roberta_logits\n    gc.collect()\n    print("E24 E0c and unchanged RoBERTa streams complete", flush=True)\n\n    modern_stream = np.zeros_like(family_score)\n    for expected_seed, path in zip(EXPECTED_SEEDS, paths, strict=True):\n        model, tokenizer, checked_metadata = load_modern_package(path, device="cpu")\n        validate_modern_metadata(path, checked_metadata, expected_seed)\n        reference = json.loads((path / "smoke_reference.json").read_text(encoding="utf-8"))\n        if reference.get("texts") != list(SMOKE_TEXTS):\n            raise ValueError(f"ModernBERT seed {expected_seed} smoke texts changed")\n        expected_smoke = np.asarray(reference.get("logits"), dtype=np.float32)\n        actual_smoke = score_modern_texts(\n            model, tokenizer, list(SMOKE_TEXTS), batch_size=2, device="cpu"\n        )\n        if expected_smoke.shape != actual_smoke.shape or not np.allclose(\n            expected_smoke, actual_smoke, rtol=0, atol=CROSS_RUNTIME_SMOKE_ATOL\n        ):\n            raise RuntimeError(f"ModernBERT seed {expected_seed} failed reload smoke check")\n        if not np.array_equal(expected_smoke.argmax(axis=1), actual_smoke.argmax(axis=1)):\n            raise RuntimeError(f"ModernBERT seed {expected_seed} changed smoke predictions")\n        logits = score_modern_texts(\n            model, tokenizer, texts, batch_size=batch_size, device="cpu"\n        )\n        modern_stream += row_standardize(logits, n_rows) / len(EXPECTED_SEEDS)\n        del model, tokenizer, logits\n        gc.collect()\n        print(f"ModernBERT seed {expected_seed} complete", flush=True)\n\n    combined = family_score + modern_stream\n    predictions = np.asarray(LABELS, dtype=object)[combined.argmax(axis=1)]\n    output = pd.DataFrame({"id": test["id"], "model": predictions})\n    if list(output.columns) != ["id", "model"] or len(output) != n_rows:\n        raise RuntimeError("Submission schema or row count changed")\n    if not output["id"].equals(test["id"]) or not set(output["model"]).issubset(LABELS):\n        raise RuntimeError("Submission IDs or labels are invalid")\n\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = output_path.with_name(output_path.name + ".tmp")\n    if temporary.exists():\n        raise ValueError(f"Stale temporary submission exists: {temporary}")\n    try:\n        output.to_parquet(temporary, index=False, engine="pyarrow")\n        roundtrip = pd.read_parquet(temporary, engine="pyarrow")\n        if not roundtrip.equals(output):\n            raise RuntimeError("Parquet round trip changed IDs or predictions")\n        temporary.replace(output_path)\n    finally:\n        if temporary.exists():\n            temporary.unlink()\n    print(f"E24 offline submission written: {output_path} ({n_rows} rows)", flush=True)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--full-package", required=True, type=Path)\n    for seed in EXPECTED_SEEDS:\n        parser.add_argument(f"--modern-seed{seed}", required=True, type=Path)\n    parser.add_argument("--e24-package", required=True, type=Path)\n    parser.add_argument("--test", required=True, type=Path)\n    parser.add_argument("--output", required=True, type=Path)\n    parser.add_argument("--modern-batch-size", type=int, default=16)\n    args = parser.parse_args()\n    submit(\n        args.full_package,\n        (args.modern_seed42, args.modern_seed17, args.modern_seed73),\n        args.e24_package,\n        args.test,\n        args.output,\n        args.modern_batch_size,\n    )\n\n\nif __name__ == "__main__":\n    main()\n', 'e39_probe_common.py': '#!/usr/bin/env python3\n"""Shared, label-free inputs and output checks for the E39 Kaggle deployment probe."""\n\nfrom __future__ import annotations\n\nimport hashlib\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\nTRAIN_SHA256 = "40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707"\nPROXY_ROWS = 4000\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef sha256_text(text: str) -> str:\n    return hashlib.sha256(text.encode("utf-8")).hexdigest()\n\n\ndef train_texts_only(path: Path) -> list[str]:\n    if sha256_file(path) != TRAIN_SHA256:\n        raise ValueError("Official training CSV hash differs from the locked protocol")\n    # The model/label columns are not loaded into the deployment probe.\n    frame = pd.read_csv(path, usecols=["text"], dtype={"text": str}, keep_default_na=False,\n                        encoding="utf-8-sig")\n    texts = frame["text"].tolist()\n    if len(texts) != 2699 or any(not text.strip() for text in texts):\n        raise ValueError("Unexpected official training text fixture")\n    return texts\n\n\ndef proxy_fixture(path: Path) -> tuple[np.ndarray, list[str]]:\n    source = train_texts_only(path)\n    texts = (source * ((PROXY_ROWS + len(source) - 1) // len(source)))[:PROXY_ROWS]\n    ids = np.asarray([f"e39-proxy-{index:05d}" for index in range(PROXY_ROWS)], dtype=str)\n    if len(set(ids)) != PROXY_ROWS or len(texts) != PROXY_ROWS:\n        raise RuntimeError("Proxy fixture is not exactly 4000 unique rows")\n    return ids, texts\n\n\ndef row_z(scores: np.ndarray, labels: tuple[str, ...], n_rows: int) -> np.ndarray:\n    array = np.asarray(scores, dtype=np.float64)\n    if array.shape != (n_rows, len(labels)) or not np.isfinite(array).all():\n        raise ValueError(f"Invalid 18-class score matrix: {array.shape}")\n    centered = array - array.mean(axis=1, keepdims=True)\n    scale = centered.std(axis=1, keepdims=True)\n    if not np.isfinite(scale).all() or np.any(scale <= 0):\n        raise ValueError("Flat or non-finite score row")\n    return centered / scale\n\n\ndef write_proxy_submission(path: Path, ids: np.ndarray, e24_scores: np.ndarray,\n                           e39_logits: np.ndarray, labels: tuple[str, ...]) -> None:\n    if path.exists():\n        raise FileExistsError(f"Probe output already exists: {path}")\n    if len(labels) != 18 or len(set(labels)) != 18:\n        raise ValueError("Official label order is not exactly 18 distinct classes")\n    if len(ids) != PROXY_ROWS or len(set(ids.tolist())) != PROXY_ROWS:\n        raise ValueError("Proxy IDs are incomplete or repeated")\n    e24_array = np.asarray(e24_scores, dtype=np.float64)\n    if e24_array.shape != (PROXY_ROWS, 18) or not np.isfinite(e24_array).all():\n        raise ValueError(f"E24 fixed score matrix is invalid: {e24_array.shape}")\n    combined = e24_array + row_z(e39_logits, labels, PROXY_ROWS)\n    if combined.shape != (PROXY_ROWS, 18) or not np.isfinite(combined).all():\n        raise ValueError("Fixed E24 + 1x E39 fusion is invalid")\n    predictions = np.asarray(labels, dtype=object)[combined.argmax(axis=1)]\n    output = pd.DataFrame({"id": ids, "model": predictions})\n    if list(output.columns) != ["id", "model"] or output.isna().any().any():\n        raise RuntimeError("Proxy submission schema is invalid")\n    if not set(output["model"]).issubset(labels):\n        raise RuntimeError("Proxy prediction contains an unknown label")\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    if temporary.exists():\n        raise FileExistsError(f"Stale temporary proxy output: {temporary}")\n    try:\n        output.to_parquet(temporary, engine="pyarrow", index=False)\n        readback = pd.read_parquet(temporary, engine="pyarrow")\n        if not readback.equals(output):\n            raise RuntimeError("Proxy Parquet round trip changed IDs or labels")\n        temporary.replace(path)\n    finally:\n        if temporary.exists():\n            temporary.unlink()\n', 'score_e24_fixed_probe.py': '#!/usr/bin/env python3\n"""Export locked E24 family scores for an unscored, text-only deployment proxy."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport e39_probe_common as common  # noqa: E402\nimport submit_e24_external_e0c as e24  # noqa: E402\n\n\ndef score(args: argparse.Namespace) -> None:\n    if args.output.exists():\n        raise FileExistsError(args.output)\n    if list(e24.MODERN_LABELS) != list(e24.LABELS):\n        raise ValueError("E24 and ModernBERT label orders differ")\n    paths = (args.modern_seed42, args.modern_seed17, args.modern_seed73)\n    resolved = tuple(path.resolve(strict=True) for path in paths)\n    if len(set(resolved)) != len(resolved):\n        raise ValueError("ModernBERT seeds must use separate packages")\n    for seed, path in zip(e24.EXPECTED_SEEDS, resolved, strict=True):\n        metadata = json.loads((path / "metadata.json").read_text(encoding="utf-8"))\n        e24.validate_modern_metadata(path, metadata, seed)\n    old_metadata = json.loads((args.full_package / "metadata.json").read_text(encoding="utf-8"))\n    for key, expected in (("train_sha256", common.TRAIN_SHA256),\n                          ("n_training_rows", 2699), ("n_training_prompts", 150)):\n        if old_metadata.get(key) != expected:\n            raise ValueError(f"E24 base package has unexpected {key}")\n\n    ids, texts = common.proxy_fixture(args.train)\n    n_rows = len(texts)\n    surface, positions = e24.load_e24_surface(args.e24_package,\n                                               Path(__file__).resolve().parents[1])\n    e24_logits = np.asarray(surface.decision_function(texts)[:, positions], dtype=np.float64)\n    _, original_e0c, roberta = e24.predict_package(args.full_package, texts, "full")\n    if original_e0c.shape != e24_logits.shape:\n        raise RuntimeError("E24 E0c and original E0c shapes differ")\n    family = e24.row_standardize(e24_logits, n_rows)\n    family += e24.row_standardize(roberta, n_rows)\n    del surface, e24_logits, original_e0c, roberta\n    gc.collect()\n\n    modern = np.zeros_like(family)\n    for seed, path in zip(e24.EXPECTED_SEEDS, resolved, strict=True):\n        model, tokenizer, metadata = e24.load_modern_package(path, device="cpu")\n        e24.validate_modern_metadata(path, metadata, seed)\n        smoke = json.loads((path / "smoke_reference.json").read_text(encoding="utf-8"))\n        if smoke.get("texts") != list(e24.SMOKE_TEXTS):\n            raise RuntimeError(f"ModernBERT seed {seed} smoke texts changed")\n        expected = np.asarray(smoke.get("logits"), dtype=np.float32)\n        actual = e24.score_modern_texts(model, tokenizer, list(e24.SMOKE_TEXTS),\n                                        batch_size=2, device="cpu")\n        if (expected.shape != actual.shape or\n                not np.allclose(expected, actual, rtol=0,\n                                atol=e24.CROSS_RUNTIME_SMOKE_ATOL) or\n                not np.array_equal(expected.argmax(axis=1), actual.argmax(axis=1))):\n            raise RuntimeError(f"ModernBERT seed {seed} failed reload smoke")\n        logits = e24.score_modern_texts(model, tokenizer, texts,\n                                        batch_size=args.modern_batch_size, device="cpu")\n        modern += e24.row_standardize(logits, n_rows) / len(e24.EXPECTED_SEEDS)\n        del model, tokenizer, logits\n        gc.collect()\n\n    combined = family + modern\n    if combined.shape != (common.PROXY_ROWS, len(e24.LABELS)) or not np.isfinite(combined).all():\n        raise RuntimeError("E24 fixed score matrix is invalid")\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(args.output, ids=ids, labels=np.asarray(e24.LABELS, dtype=str),\n                        scores=combined)\n    print(f"E24_FIXED_PROXY_SCORES_READY rows={len(ids)}", flush=True)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--full-package", type=Path, required=True)\n    parser.add_argument("--modern-seed42", type=Path, required=True)\n    parser.add_argument("--modern-seed17", type=Path, required=True)\n    parser.add_argument("--modern-seed73", type=Path, required=True)\n    parser.add_argument("--e24-package", type=Path, required=True)\n    parser.add_argument("--train", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--modern-batch-size", type=int, default=16)\n    args = parser.parse_args()\n    if args.modern_batch_size < 1:\n        raise ValueError("Batch size must be positive")\n    score(args)\n\n\nif __name__ == "__main__":\n    main()\n', 'probe_e39_gpu.py': '#!/usr/bin/env python3\n"""Unscored E39 fold-1 adapter load and 4000-text Kaggle T4 deployment probe.\n\n`reference` runs once on the training server against three fold-1 TRAIN-side\ntexts. `environment`, `smoke`, and `full` run in an Internet-off Kaggle GPU\nNotebook. This file never reads a target label or computes accuracy.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport importlib.metadata\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport e39_probe_common as common  # noqa: E402\n\n\nMAX_LENGTH = 256\nSMOKE_ATOL = 0.05  # Fixed before any Kaggle T4 run; argmax must also agree.\nSMOKE_TRAIN_ROW_INDICES = (1, 3, 4)\nE0_OOF_SHA256 = "698265e92c50f23acc6eb68731721249a27a5966e30b62c4eed3c0a30c3528fa"\nOFFICIAL_LABELS = (\n    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",\n    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash",\n    "gemini-3.8-flash", "claude-fable-5", "claude-opus-4-6",\n    "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5", "qwen3.8-max",\n    "glm-5.3", "glm-5.3-flash", "kimi-k3",\n)\nMIN_T4_MARGIN_BYTES = 256 * 2**20\nMAX_FULL_INFERENCE_SECONDS = 3 * 3600\nRUNTIME = {\n    "torch": "2.6.0", "transformers": "4.51.3", "peft": "0.15.2",\n    "accelerate": "1.6.0", "bitsandbytes": "0.45.5",\n}\n\n\ndef read_manifest(path: Path) -> dict:\n    document = json.loads(path.read_text(encoding="utf-8"))\n    if (document.get("format_version") != 1 or document.get("fold") != 1\n            or document.get("model_repo") != "Qwen/Qwen3-8B-Base"\n            or document.get("model_revision") != "49e3418fbbbca6ecbdf9608b4d22e5a407081db4"\n            or document.get("train_sha256") != common.TRAIN_SHA256\n            or document.get("labels") != list(OFFICIAL_LABELS)\n            or len(document.get("model_files", {})) != 12\n            or "adapter_model.safetensors" not in document.get("adapter_files", {})\n            or document.get("purpose") != "unscored_kaggle_t4_deployment_probe_only"):\n        raise ValueError("E39 deployment asset manifest differs from locked fold 1")\n    return document\n\n\ndef verify_assets(model_dir: Path, adapter_dir: Path, manifest: dict) -> None:\n    for directory, entries in ((model_dir, manifest["model_files"]),\n                               (adapter_dir, manifest["adapter_files"])):\n        root = directory.resolve(strict=True)\n        for name, value in entries.items():\n            relative = Path(name)\n            if relative.name != name or name.startswith(".") or ".." in relative.parts:\n                raise ValueError(f"Unsafe asset name: {name}")\n            path = directory / name\n            if path.is_symlink() or not path.is_file() or not path.resolve().is_relative_to(root):\n                raise ValueError(f"Missing or redirected E39 asset: {name}")\n            expected = value["sha256"] if isinstance(value, dict) else value\n            if isinstance(value, dict) and path.stat().st_size != value["size"]:\n                raise ValueError(f"E39 asset size changed: {name}")\n            if common.sha256_file(path) != expected:\n                raise ValueError(f"E39 asset SHA-256 changed: {name}")\n    config = json.loads((adapter_dir / "adapter_config.json").read_text(encoding="utf-8"))\n    if (config.get("r") != 16 or config.get("lora_alpha") != 32\n            or config.get("task_type") != "SEQ_CLS"):\n        raise ValueError("E39 fold adapter configuration differs from locked training")\n    base_config = json.loads((model_dir / "config.json").read_text(encoding="utf-8"))\n    if base_config.get("model_type") != "qwen3" or base_config.get("num_hidden_layers") != 36:\n        raise ValueError("E39 pinned Base architecture differs from 36-layer Qwen3")\n\n\ndef package_versions(stage: str) -> dict[str, str]:\n    versions = {name: importlib.metadata.version(name) for name in RUNTIME}\n    if {key: value.split("+")[0] for key, value in versions.items()} != RUNTIME:\n        raise RuntimeError(f"GPU runtime differs from E39 pinned packages: {versions}")\n    if stage != "reference":\n        import io\n\n        arrow_version = importlib.metadata.version("pyarrow")\n        if not 15 <= int(arrow_version.split(".")[0]) < 30:\n            raise RuntimeError(f"Kaggle GPU kernel pyarrow is outside the probe range: {arrow_version}")\n        buffer = io.BytesIO()\n        trial = pd.DataFrame({"id": ["schema-probe"], "model": ["schema-probe"]})\n        trial.to_parquet(buffer, engine="pyarrow", index=False)\n        buffer.seek(0)\n        if not pd.read_parquet(buffer, engine="pyarrow").equals(trial):\n            raise RuntimeError("Kaggle GPU kernel cannot round-trip Parquet")\n        versions["pyarrow"] = arrow_version\n    return versions\n\n\ndef check_hardware(stage: str) -> dict:\n    import torch\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("A CUDA accelerator is required for E39 deployment probe")\n    devices = []\n    for index in range(torch.cuda.device_count()):\n        info = torch.cuda.get_device_properties(index)\n        devices.append({"index": index, "name": info.name, "total_bytes": info.total_memory})\n    if stage == "reference":\n        if len(devices) != 1 or "3090" not in devices[0]["name"]:\n            raise RuntimeError("The smoke reference must use one server RTX 3090")\n    else:\n        if (len(devices) != 2 or any("T4" not in row["name"] for row in devices)\n                or any(not (14 * 2**30 <= row["total_bytes"] <= 17 * 2**30)\n                       for row in devices)):\n            raise RuntimeError(f"The deployment probe requires Kaggle T4 x2: {devices}")\n    return {"devices": devices, "torch_cuda": torch.version.cuda}\n\n\ndef device_map(mode: str) -> dict:\n    if mode == "single":\n        return {"": 0}\n    if mode != "dual":\n        raise ValueError("Device mode must be single or dual")\n    # Qwen3-8B-Base has 36 decoder layers. This fixed engineering map is used\n    # only if the single T4 probe cannot load; it is never chosen by scores.\n    mapping = {"model.embed_tokens": 0}\n    mapping.update({f"model.layers.{index}": 0 if index < 18 else 1\n                    for index in range(36)})\n    mapping.update({"model.norm": 1, "score": 1})\n    return mapping\n\n\ndef load_model(model_dir: Path, adapter_dir: Path, labels: tuple[str, ...], mode: str):\n    import torch\n    from peft import PeftModel\n    from transformers import (AutoModelForSequenceClassification, AutoTokenizer,\n                              BitsAndBytesConfig)\n\n    tokenizer = AutoTokenizer.from_pretrained(adapter_dir, local_files_only=True,\n                                               trust_remote_code=False, use_fast=True)\n    tokenizer.padding_side = "right"\n    if tokenizer.pad_token_id is None:\n        raise RuntimeError("E39 tokenizer lacks its pinned pad token")\n    quantization = BitsAndBytesConfig(\n        load_in_4bit=True, bnb_4bit_quant_type="nf4",\n        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,\n    )\n    base = AutoModelForSequenceClassification.from_pretrained(\n        model_dir, local_files_only=True, trust_remote_code=False,\n        num_labels=len(labels), id2label=dict(enumerate(labels)),\n        label2id={label: index for index, label in enumerate(labels)},\n        quantization_config=quantization, torch_dtype=torch.float16,\n        device_map=device_map(mode),\n    )\n    base.config.pad_token_id = tokenizer.pad_token_id\n    model = PeftModel.from_pretrained(base, adapter_dir, is_trainable=False)\n    model.eval()\n    return model, tokenizer\n\n\ndef sample_actual_gpu_free(minimum: list[int]) -> None:\n    """Record driver-visible free bytes, including non-PyTorch allocations."""\n    import torch\n\n    for index in range(len(minimum)):\n        free, _ = torch.cuda.mem_get_info(index)\n        minimum[index] = min(minimum[index], free)\n\n\ndef score_texts(model, tokenizer, texts: list[str], labels: tuple[str, ...],\n                minimum_actual_free: list[int],\n                timeout_seconds: float | None = None) -> np.ndarray:\n    import torch\n\n    result = np.empty((len(texts), len(labels)), dtype=np.float32)\n    began = time.monotonic()\n    with torch.inference_mode():\n        for index, text in enumerate(texts):\n            if timeout_seconds is not None and time.monotonic() - began > timeout_seconds:\n                raise TimeoutError("E39 4000-text GPU proxy exceeded three hours")\n            encoded = tokenizer(text, truncation=True, max_length=MAX_LENGTH,\n                                return_tensors="pt", add_special_tokens=True)\n            encoded = {name: value.to("cuda:0") for name, value in encoded.items()}\n            logits = model(**encoded).logits\n            if logits.shape != (1, len(labels)) or not torch.isfinite(logits).all():\n                raise FloatingPointError(f"Invalid E39 logits at proxy row {index}")\n            sample_actual_gpu_free(minimum_actual_free)\n            result[index] = logits.float().cpu().numpy()[0]\n            sample_actual_gpu_free(minimum_actual_free)\n    for device in range(torch.cuda.device_count()):\n        torch.cuda.synchronize(device)\n    if not np.isfinite(result).all():\n        raise FloatingPointError("Non-finite E39 proxy scores")\n    return result\n\n\ndef memory_report(minimum_actual_free: list[int]) -> list[dict]:\n    import torch\n\n    rows = []\n    for index in range(torch.cuda.device_count()):\n        total = torch.cuda.get_device_properties(index).total_memory\n        reserved = torch.cuda.max_memory_reserved(index)\n        free, _ = torch.cuda.mem_get_info(index)\n        rows.append({"index": index, "peak_reserved_bytes": reserved,\n                     "total_bytes": total, "remaining_at_peak_bytes": total - reserved,\n                     "current_actual_free_bytes": free,\n                     "minimum_actual_free_bytes": min(free, minimum_actual_free[index])})\n    return rows\n\n\ndef reference_rows(train_path: Path, e0_oof: Path) -> tuple[list[int], list[str]]:\n    texts = common.train_texts_only(train_path)\n    if common.sha256_file(e0_oof) != E0_OOF_SHA256:\n        raise ValueError("E0 five-fold assignment file differs from the locked protocol")\n    frame = pd.read_csv(train_path, usecols=["id", "text"], dtype=str,\n                        keep_default_na=False, encoding="utf-8-sig")\n    if frame["text"].tolist() != texts or frame["id"].duplicated().any():\n        raise ValueError("Official training row order changed")\n    oof = pd.read_csv(e0_oof, usecols=["id", "fold"], dtype=str, keep_default_na=False)\n    if len(oof) != 2160 or oof["id"].duplicated().any() or not oof["fold"].isin(\n            ["1", "2", "3", "4", "5"]).all():\n        raise ValueError("E0 development OOF split is invalid")\n    training_ids = set(oof.loc[oof["fold"] != "1", "id"])\n    positions = [index for index, item in enumerate(frame["id"]) if item in training_ids][:3]\n    if tuple(positions) != SMOKE_TRAIN_ROW_INDICES:\n        raise RuntimeError("Fold-1 training-side smoke rows differ from preregistration")\n    return positions, [texts[index] for index in positions]\n\n\ndef run(args: argparse.Namespace) -> None:\n    import torch\n\n    started = time.monotonic()\n    versions = package_versions(args.stage)\n    hardware = check_hardware(args.stage)\n    manifest = read_manifest(args.manifest)\n    labels = tuple(manifest["labels"])\n    if args.stage == "environment":\n        print(json.dumps({"stage": args.stage, "packages": versions,\n                          "hardware": hardware, "internet_required": False}, indent=2), flush=True)\n        return\n    verify_assets(args.model_dir, args.adapter_dir, manifest)\n    minimum_actual_free = [torch.cuda.mem_get_info(index)[0]\n                           for index in range(torch.cuda.device_count())]\n    for index in range(torch.cuda.device_count()):\n        torch.cuda.reset_peak_memory_stats(index)\n    model, tokenizer = load_model(args.model_dir, args.adapter_dir, labels, args.device_mode)\n    sample_actual_gpu_free(minimum_actual_free)\n    loaded_seconds = time.monotonic() - started\n    if args.stage == "reference":\n        positions, texts = reference_rows(args.train, args.e0_oof)\n        logits = score_texts(model, tokenizer, texts, labels, minimum_actual_free)\n        if args.output.exists():\n            raise FileExistsError(args.output)\n        report = {"format_version": 1, "fold": 1, "labels": labels,\n                  "model_revision": manifest["model_revision"],\n                  "train_sha256": common.TRAIN_SHA256,\n                  "base_manifest_sha256": common.sha256_file(args.manifest),\n                  "adapter_weights_sha256": manifest["adapter_files"]["adapter_model.safetensors"],\n                  "max_abs_logit_difference_allowed": SMOKE_ATOL,\n                  "rows": [{"train_row_index": position,\n                            "text_sha256": common.sha256_text(text),\n                            "logits": logits[index].astype(float).tolist(),\n                            "argmax": int(logits[index].argmax())}\n                           for index, (position, text) in enumerate(zip(positions, texts, strict=True))],\n                  "packages": versions, "hardware": hardware,\n                  "memory": memory_report(minimum_actual_free),\n                  "seconds": time.monotonic() - started}\n        args.output.parent.mkdir(parents=True, exist_ok=True)\n        args.output.write_text(json.dumps(report, indent=2) + "\\n", encoding="utf-8")\n        print("E39_TRAIN_SIDE_SMOKE_REFERENCE_READY", flush=True)\n        return\n\n    reference = json.loads(args.smoke_reference.read_text(encoding="utf-8"))\n    if (reference.get("format_version") != 1 or reference.get("fold") != 1\n            or reference.get("labels") != list(labels)\n            or reference.get("train_sha256") != common.TRAIN_SHA256\n            or reference.get("base_manifest_sha256") != common.sha256_file(args.manifest)\n            or reference.get("adapter_weights_sha256") !=\n            manifest["adapter_files"]["adapter_model.safetensors"]\n            or reference.get("max_abs_logit_difference_allowed") != SMOKE_ATOL\n            or len(reference.get("rows", [])) != 3):\n        raise ValueError("E39 training-side smoke reference does not match the locked assets")\n    source_texts = common.train_texts_only(args.train)\n    selected = []\n    for item in reference["rows"]:\n        ordinal = item["train_row_index"]\n        if not isinstance(ordinal, int) or not 0 <= ordinal < len(source_texts):\n            raise ValueError("Smoke row ordinal is invalid")\n        text = source_texts[ordinal]\n        if common.sha256_text(text) != item["text_sha256"]:\n            raise ValueError("Smoke train-side text hash differs")\n        selected.append(text)\n    actual = score_texts(model, tokenizer, selected, labels, minimum_actual_free)\n    expected = np.asarray([item["logits"] for item in reference["rows"]], dtype=np.float32)\n    if (expected.shape != (3, 18) or not np.isfinite(expected).all()\n            or not np.allclose(actual, expected, atol=SMOKE_ATOL, rtol=0)\n            or not np.array_equal(actual.argmax(axis=1), expected.argmax(axis=1))):\n        raise RuntimeError("E39 T4 smoke logits differ from 3090 fold-1 reference")\n    if args.stage == "smoke":\n        if args.output.exists():\n            raise FileExistsError(args.output)\n        report = {"stage": "smoke", "passed": True, "device_mode": args.device_mode,\n                  "model_manifest_sha256": common.sha256_file(args.manifest),\n                  "smoke_reference_sha256": common.sha256_file(args.smoke_reference),\n                  "packages": versions, "hardware": hardware,\n                  "load_seconds": loaded_seconds, "total_seconds": time.monotonic() - started,\n                  "memory": memory_report(minimum_actual_free), "logit_atol": SMOKE_ATOL,\n                  "argmax_equal": True}\n        if any(row["minimum_actual_free_bytes"] < MIN_T4_MARGIN_BYTES\n               for row in report["memory"]):\n            raise MemoryError("E39 T4 smoke actual free memory fell below 256 MiB")\n        args.output.parent.mkdir(parents=True, exist_ok=True)\n        args.output.write_text(json.dumps(report, indent=2) + "\\n", encoding="utf-8")\n        print("E39_T4_SMOKE_PASS", flush=True)\n        return\n\n    if args.stage != "full":\n        raise ValueError(args.stage)\n    smoke = json.loads(args.smoke_report.read_text(encoding="utf-8"))\n    if (smoke.get("stage") != "smoke" or smoke.get("passed") is not True\n            or smoke.get("device_mode") != args.device_mode\n            or smoke.get("model_manifest_sha256") != common.sha256_file(args.manifest)\n            or smoke.get("smoke_reference_sha256") != common.sha256_file(args.smoke_reference)):\n        raise ValueError("A matching successful T4 smoke report is required before full proxy")\n    ids, texts = common.proxy_fixture(args.train)\n    inference_started = time.monotonic()\n    logits = score_texts(model, tokenizer, texts, labels, minimum_actual_free,\n                         timeout_seconds=MAX_FULL_INFERENCE_SECONDS)\n    inference_seconds = time.monotonic() - inference_started\n    memory = memory_report(minimum_actual_free)\n    if any(row["minimum_actual_free_bytes"] < MIN_T4_MARGIN_BYTES for row in memory):\n        raise MemoryError("E39 4000-row T4 actual free memory fell below 256 MiB")\n    if args.output.exists():\n        raise FileExistsError(args.output)\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(args.output, ids=ids, labels=np.asarray(labels, dtype=str),\n                        logits=logits)\n    summary = {"stage": "full_unscored_proxy", "rows": len(ids),\n               "device_mode": args.device_mode,\n               "model_manifest_sha256": common.sha256_file(args.manifest),\n               "smoke_reference_sha256": common.sha256_file(args.smoke_reference),\n               "packages": versions, "hardware": hardware, "memory": memory,\n               "load_seconds": loaded_seconds, "inference_seconds": inference_seconds,\n               "total_seconds": time.monotonic() - started,\n               "validation_scores_computed": False,\n               "labels_loaded": False}\n    args.output.with_suffix(".json").write_text(json.dumps(summary, indent=2) + "\\n",\n                                                encoding="utf-8")\n    print("E39_T4_FULL_PROXY_READY rows=4000", flush=True)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("stage", choices=["environment", "reference", "smoke", "full"])\n    parser.add_argument("--manifest", type=Path, required=True)\n    parser.add_argument("--model-dir", type=Path)\n    parser.add_argument("--adapter-dir", type=Path)\n    parser.add_argument("--train", type=Path)\n    parser.add_argument("--e0-oof", type=Path)\n    parser.add_argument("--smoke-reference", type=Path)\n    parser.add_argument("--smoke-report", type=Path)\n    parser.add_argument("--output", type=Path)\n    parser.add_argument("--device-mode", choices=["single", "dual"], default="single")\n    args = parser.parse_args()\n    if args.stage != "environment":\n        for name in ("model_dir", "adapter_dir", "train", "output"):\n            if getattr(args, name) is None:\n                parser.error(f"--{name.replace(\'_\', \'-\')} is required for {args.stage}")\n    if args.stage == "reference" and args.e0_oof is None:\n        parser.error("--e0-oof is required for reference")\n    if args.stage in ("smoke", "full") and args.smoke_reference is None:\n        parser.error("--smoke-reference is required")\n    if args.stage == "full" and args.smoke_report is None:\n        parser.error("--smoke-report is required for full")\n    run(args)\n\n\nif __name__ == "__main__":\n    main()\n', 'e39_probe_asset_manifest.json': '{\n  "format_version": 1,\n  "experiment": "e39_qwen3_8b_base_official_only_fivefold",\n  "fold": 1,\n  "model_repo": "Qwen/Qwen3-8B-Base",\n  "model_revision": "49e3418fbbbca6ecbdf9608b4d22e5a407081db4",\n  "train_sha256": "40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707",\n  "labels": [\n    "gpt-5.5",\n    "gpt-5.6-luna",\n    "gpt-5.6-sol",\n    "gpt-5.6-terra",\n    "gpt-6-astra",\n    "gemini-3.1-pro-preview",\n    "gemini-3.6-flash",\n    "gemini-3.7-flash",\n    "gemini-3.8-flash",\n    "claude-fable-5",\n    "claude-opus-4-6",\n    "claude-opus-4-8",\n    "claude-opus-5",\n    "claude-sonnet-5",\n    "qwen3.8-max",\n    "glm-5.3",\n    "glm-5.3-flash",\n    "kimi-k3"\n  ],\n  "model_files": {\n    "config.json": {\n      "size": 729,\n      "sha256": "3bd01d7ad7a2e203ecbbe84e24087a51c6d2a108ee4bcc42d0016bf49564983a"\n    },\n    "generation_config.json": {\n      "size": 138,\n      "sha256": "8c970692323e3ea0e9b8b0a4dca79388d31226e41f83c9fd6014804280ebf6e8"\n    },\n    "merges.txt": {\n      "size": 1671853,\n      "sha256": "8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5"\n    },\n    "model-00001-of-00005.safetensors": {\n      "size": 3996250744,\n      "sha256": "9983f1b9ef2f60e7c3730d9bc11ada914e6ec630639b5b020a89bd158cd0446b"\n    },\n    "model-00002-of-00005.safetensors": {\n      "size": 3993160032,\n      "sha256": "9aa12339835bf7a093d3d0b0a0d2d77f8538301cfc7f8e7ec7a585858ebf7a1f"\n    },\n    "model-00003-of-00005.safetensors": {\n      "size": 3959604768,\n      "sha256": "c7dd5a191c8da555def0714550375b3c0117751add408fcdda0c2a79ac0186bc"\n    },\n    "model-00004-of-00005.safetensors": {\n      "size": 3187841392,\n      "sha256": "ad8b708792105133e03fe19e2d56c89c709f677f01285f8b3ede8947d59fd9fe"\n    },\n    "model-00005-of-00005.safetensors": {\n      "size": 1244659840,\n      "sha256": "fbf24915d47ea030bb68ab0b9488f4515a907185baa6dc26837c9c3f2326a550"\n    },\n    "model.safetensors.index.json": {\n      "size": 32878,\n      "sha256": "cd39d5e44c5aadfd1954b598883aea1c4a98822a2f7723d214ef2278b7fec2e4"\n    },\n    "tokenizer.json": {\n      "size": 7031645,\n      "sha256": "c0382117ea329cdf097041132f6d735924b697924d6f6fc3945713e96ce87539"\n    },\n    "tokenizer_config.json": {\n      "size": 9678,\n      "sha256": "3c04ed3ca964ea2f6b2b5faf0dc4d31aec1cb1e8b4bcf63f402d295046b422b5"\n    },\n    "vocab.json": {\n      "size": 2776833,\n      "sha256": "ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910"\n    }\n  },\n  "adapter_files": {\n    "special_tokens_map.json": "6676f091c8bc4d1b50146427cfde92073402866b87b6e39223227931b70083e9",\n    "tokenizer.json": "f567e68cbda9ddc0fa93307e621455e1dd174cc9f5a1c4228c40b5791d68052e",\n    "adapter_model.safetensors": "576ac1ff2a26c0bed24a56db90a47fc4e13f7e8596426e95cef7dd5eb8046354",\n    "merges.txt": "8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5",\n    "tokenizer_config.json": "9a3a6f623ddaad552d27957966270df1c7c6b8777ddaebeb4753abd4ed635edd",\n    "adapter_config.json": "5023c949affa666fbd0ea4abe3f1e7eace584e236addff3df40118a477c8d0ec",\n    "README.md": "64ae5768344daf71cf252d753bb12eaa9fb32ac8120f8cad33d72d0be256adc4",\n    "added_tokens.json": "c0284b582e14987fbd3d5a2cb2bd139084371ed9acbae488829a1c900833c680",\n    "vocab.json": "ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910"\n  },\n  "purpose": "unscored_kaggle_t4_deployment_probe_only"\n}\n'}
SOURCE_SHA256 = {'submit_e24_external_e0c.py': 'da06935ebfd5963c7e6c9a81dc5515bf60438d3cbe876db5325b02745912d8f2', 'e39_probe_common.py': '025dbea973d61bcb539f25a6807e2a64f78bfe918dd8e4a18cc6b26399f23056', 'score_e24_fixed_probe.py': '8a96f481ad81483f7cc1a8d3207663119c2b7f5e4531fbde70edb9692ffc1ac6', 'probe_e39_gpu.py': '602e1417c05f48ede2aad12ce170eec8e02cbbbdf5b159a077104d1c64e6da97', 'e39_probe_asset_manifest.json': 'd2e381f58249ce26eb9cfc1dc993a48b8386c1a6fce91bed058e90d37068f018'}
for name, value in SOURCE_TEXT.items():
    target = WORK_KAGGLE / name
    target.write_text(value, encoding='utf-8')
    if hashlib.sha256(target.read_bytes()).hexdigest() != SOURCE_SHA256[name]:
        raise RuntimeError(f'E39 probe source hash mismatch: {name}')
WORK_MANIFEST = WORK_KAGGLE / 'e39_probe_asset_manifest.json'
GPU_SCRIPT = WORK_KAGGLE / 'probe_e39_gpu.py'
CPU_SCRIPT = WORK_KAGGLE / 'score_e24_fixed_probe.py'
if DEVICE_MODE not in ('single', 'dual'):
    raise ValueError('DEVICE_MODE must be single or dual')
PROBE_ROOT.mkdir(parents=True, exist_ok=False)
print('E39 probe source and asset manifest hashes verified')


In [ ]:
import json

asset_manifest = json.loads(WORK_MANIFEST.read_text(encoding='utf-8'))
def one_match(paths, kind):
    candidates = sorted(set(path.resolve(strict=True) for path in paths))
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one {kind} directory, found {len(candidates)}; set it explicitly')
    return candidates[0]

if BASE_DIR is None:
    expected = asset_manifest['model_files']['config.json']['sha256']
    BASE_DIR = one_match((p.parent for p in INPUT_ROOT.rglob('config.json')
                          if sha256_file(p) == expected), 'pinned Qwen3-8B-Base')
else:
    BASE_DIR = Path(BASE_DIR).resolve(strict=True)
if ADAPTER_DIR is None:
    expected = asset_manifest['adapter_files']['adapter_model.safetensors']
    ADAPTER_DIR = one_match((p.parent for p in INPUT_ROOT.rglob('adapter_model.safetensors')
                             if sha256_file(p) == expected), 'private E39 fold-1 adapter')
else:
    ADAPTER_DIR = Path(ADAPTER_DIR).resolve(strict=True)
if SMOKE_REFERENCE_PATH is None:
    paths = list(INPUT_ROOT.rglob('e39_fold1_smoke_reference.json'))
    if len(paths) != 1:
        raise RuntimeError('Attach exactly one private train-side smoke reference')
    SMOKE_REFERENCE_PATH = paths[0]
SMOKE_REFERENCE_PATH = Path(SMOKE_REFERENCE_PATH).resolve(strict=True)

GPU_ENV = os.environ.copy()
GPU_ENV.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', HF_HUB_DISABLE_XET='1',
               TOKENIZERS_PARALLELISM='false', PYTHONNOUSERSITE='1')
GPU_PYTHON = Path(GPU_PYTHON).resolve(strict=True)
subprocess.run([str(GPU_PYTHON), str(GPU_SCRIPT), 'environment',
                '--manifest', str(WORK_MANIFEST)], cwd=WORK_SOURCE,
               env=GPU_ENV, check=True, timeout=300)
print('E39 GPU environment checked; assets will be hashed before load')


In [ ]:
import numpy as np
import pandas as pd

TRAIN_PATH = COMPETITION_DIR / 'train.csv'
if not TRAIN_PATH.is_file():
    raise FileNotFoundError('Official train.csv must be attached for the text-only proxy')
SMOKE_REPORT = PROBE_ROOT / 'smoke.json'
GPU_SCORES = PROBE_ROOT / 'e39_proxy_logits.npz'
CPU_SCORES = PROBE_ROOT / 'e24_fixed_scores.npz'
PROXY_PARQUET = PROBE_ROOT / 'submission.parquet'
gpu_args = ['--manifest', str(WORK_MANIFEST), '--model-dir', str(BASE_DIR),
            '--adapter-dir', str(ADAPTER_DIR), '--train', str(TRAIN_PATH),
            '--smoke-reference', str(SMOKE_REFERENCE_PATH), '--device-mode', DEVICE_MODE]
subprocess.run([str(GPU_PYTHON), str(GPU_SCRIPT), 'smoke', *gpu_args,
                '--output', str(SMOKE_REPORT)], cwd=WORK_SOURCE, env=GPU_ENV,
               check=True, timeout=3600)

CPU_ENV = os.environ.copy()
CPU_ENV.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1',
               HF_HUB_DISABLE_XET='1', TOKENIZERS_PARALLELISM='false',
               TORCHDYNAMO_DISABLE='1', CUDA_VISIBLE_DEVICES='',
               OMP_NUM_THREADS=str(min(4, os.cpu_count() or 4)),
               MKL_NUM_THREADS=str(min(4, os.cpu_count() or 4)))
subprocess.run([
    str(PYTHON_EXECUTABLE), str(CPU_SCRIPT),
    '--full-package', str(PACKAGE_DIR),
    '--modern-seed42', str(MODERN_PACKAGE_DIRS[42]),
    '--modern-seed17', str(MODERN_PACKAGE_DIRS[17]),
    '--modern-seed73', str(MODERN_PACKAGE_DIRS[73]),
    '--e24-package', str(E24_PACKAGE_DIR),
    '--train', str(TRAIN_PATH), '--output', str(CPU_SCORES),
], cwd=WORK_SOURCE, env=CPU_ENV, check=True, timeout=4 * 3600)

subprocess.run([str(GPU_PYTHON), str(GPU_SCRIPT), 'full', *gpu_args,
                '--smoke-report', str(SMOKE_REPORT), '--output', str(GPU_SCORES)],
               cwd=WORK_SOURCE, env=GPU_ENV, check=True, timeout=4 * 3600)

sys.path.insert(0, str(WORK_KAGGLE))
import e39_probe_common as common
with np.load(CPU_SCORES, allow_pickle=False) as e24_data, np.load(GPU_SCORES, allow_pickle=False) as e39_data:
    ids = e24_data['ids']
    if not np.array_equal(ids, e39_data['ids']):
        raise RuntimeError('E24 and E39 proxy ID order differs')
    labels = tuple(e24_data['labels'].tolist())
    if labels != tuple(e39_data['labels'].tolist()) or labels != tuple(asset_manifest['labels']):
        raise RuntimeError('E24 and E39 official label order differs')
    common.write_proxy_submission(PROXY_PARQUET, ids, e24_data['scores'],
                                  e39_data['logits'], labels)
elapsed = time.monotonic() - PROBE_STARTED
if elapsed > TIME_LIMIT_SECONDS:
    raise TimeoutError(f'Full E39 deployment proxy exceeded eight-hour safety budget: {elapsed:.1f}s')
roundtrip = pd.read_parquet(PROXY_PARQUET, engine='pyarrow')
if list(roundtrip.columns) != ['id', 'model'] or len(roundtrip) != 4000:
    raise RuntimeError('Proxy submission Parquet schema/row count changed')
print('E39_UNSCORED_T4_DEPLOYMENT_PROXY_PASS', len(roundtrip), round(elapsed, 1))
print('Do not submit this synthetic-ID proxy Notebook to the competition.')
